# 03 — Feature Engineering

This notebook continues the workflow from:

- **01_load_data.ipynb** (loading raw `.txt` sensor files)
- **02_eda.ipynb** (statistical and visual exploration)

Here we generate engineered features used later in **04_modeling.ipynb**.


In [1]:
# --- STEP 1. Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

# --- STEP 2. Navigate to your project root ---
project_path = '/content/drive/MyDrive/hydraulic_dashboard'

import os
os.chdir(project_path)
print('Current working directory:', os.getcwd())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current working directory: /content/drive/MyDrive/hydraulic_dashboard


## 1. Pipeline Context and Dependencies

### From Notebook 01 (`01_load_data.ipynb`)
- Loaded raw `.txt` sensor data
- Performed basic cleaning and alignment
- Saved canonical artifacts:
  - `data/processed/X_features.parquet`
  - `data/processed/y_labels.parquet`
  - `data/metadata/features_index.csv`

### From Notebook 02 (`02_eda.ipynb`)
- Checked shape, missingness, and sensor ranges
- Identified constant and sparse spike-like features
- Confirmed that spikes are physical events, not noise
- Verified class balance and stability behaviour

This notebook builds on those artifacts to prepare the final feature matrix for modeling.


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

BASE_DIR = Path.cwd().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd()
DATA_DIR = BASE_DIR / "data"
PROC_DIR = DATA_DIR / "processed"
META_DIR = DATA_DIR / "metadata"

OUT_PATH = PROC_DIR / "X_features_fe.parquet"

print("BASE_DIR :", BASE_DIR)
print("PROC_DIR :", PROC_DIR)
print("META_DIR :", META_DIR)
print("Output will be saved to:", OUT_PATH)


BASE_DIR : /content/drive/MyDrive/hydraulic_dashboard
PROC_DIR : /content/drive/MyDrive/hydraulic_dashboard/data/processed
META_DIR : /content/drive/MyDrive/hydraulic_dashboard/data/metadata
Output will be saved to: /content/drive/MyDrive/hydraulic_dashboard/data/processed/X_features_fe.parquet


## 2. Objectives of Feature Engineering

We aim to produce the final ML-ready dataset for the modeling stage.

### Key Goals
- Clean and align sensor readings
- Remove constant (zero-variance) features identified in 02_eda.ipynb
- Respect sparse spike-like features as physical events (keep them for modeling)
- Add statistical features
- Add time-based (delta, rolling) features
- Add group-level summary signals
- Ensure deterministic reproducibility

### Outputs
- **X_features_fe.parquet**
- **y_labels.parquet**


In [3]:
X = pd.read_parquet(PROC_DIR / "X_features.parquet")
y = pd.read_parquet(PROC_DIR / "y_labels.parquet")
features_index = pd.read_csv(META_DIR / "features_index.csv")

print("X shape:", X.shape)
print("y shape:", y.shape)
print("features_index shape:", features_index.shape)

display(X.head(3))
display(y.head(3))
display(features_index.head(3))

assert len(X) == len(y)
assert {"column","sensor","group"}.issubset(features_index.columns)


X shape: (2205, 43680)
y shape: (2205, 5)
features_index shape: (43680, 3)


,CE_t1,CE_t2,CE_t3,CE_t4,CE_t5,CE_t6,CE_t7,CE_t8,CE_t9,CE_t10,...,VS1_t51,VS1_t52,VS1_t53,VS1_t54,VS1_t55,VS1_t56,VS1_t57,VS1_t58,VS1_t59,VS1_t60
0,47.202,47.273,47.250,47.332,47.213,47.372,47.273,47.438,46.691,46.599,...,0.554,0.552,0.545,0.553,0.553,0.539,0.544,0.545,0.535,0.543
1,29.208,28.822,28.805,28.922,28.591,28.643,28.216,27.812,27.514,27.481,...,0.555,0.547,0.548,0.544,0.536,0.542,0.540,0.533,0.531,0.534
2,23.554,23.521,23.527,23.008,23.042,23.052,22.658,22.952,22.908,22.359,...,0.543,0.544,0.543,0.554,0.544,0.544,0.545,0.544,0.530,0.534


,cooler_condition,valve_condition,internal_pump_leakage,hydraulic_accumulator,stable_flag
0,3,100,0,130,1
1,3,100,0,130,1
2,3,100,0,130,1


,column,sensor,group
0,CE_t1,CE,cooler
1,CE_t2,CE,cooler
2,CE_t3,CE,cooler


In [4]:
# 2.x) Reuse EDA findings: drop constant features, keep spike-like features

numeric = X.select_dtypes(include=[np.number])
summary_stats = numeric.describe().T

# Constant features: zero variance (std == 0)
constant_mask = summary_stats["std"].fillna(0) == 0
constant_cols = summary_stats.index[constant_mask].tolist()
print(f"Found {len(constant_cols)} constant features with std=0.")

# Sparse "spike" features (physically meaningful, DO NOT DROP)
sparse_mask = (
    (summary_stats["25%"] == 0) &
    (summary_stats["50%"] == 0) &
    (summary_stats["75%"] == 0) &
    (summary_stats["max"] > 0)
)
sparse_cols = summary_stats.index[sparse_mask].tolist()
print(f"Found {len(sparse_cols)} sparse spike-like features (kept).")

# Drop only constant features from X and keep features_index aligned
if constant_cols:
    X = X.drop(columns=constant_cols)
    features_index = (
        features_index
        .loc[~features_index["column"].isin(constant_cols)]
        .reset_index(drop=True)
    )

print("X shape after dropping constants:", X.shape)
print("features_index shape after alignment:", features_index.shape)


Found 34 constant features with std=0.
Found 1681 sparse spike-like features (kept).
X shape after dropping constants: (2205, 43646)
features_index shape after alignment: (43646, 3)


## 3. Feature Engineering Overview

### 3.1 Core Transformations
- Base features (already cleaned in 01)
- Rolling aggregations
- First differences
- Cycle-level summaries

### 3.2 Rolling and Delta Features
- Capture local temporal dynamics around each cycle
- Smooth out noise without destroying physical spikes

### 3.3 Group Features
- Use `features_index.csv` to aggregate sensors into functional groups
- Produce group-level mean / std / min / max per cycle

All of these are appended horizontally to form `X_features_fe.parquet`.


In [5]:
def add_rolling_features(df: pd.DataFrame, window: int = 5, suffix: str = "roll5") -> pd.DataFrame:
    """Add simple rolling mean features per numeric column.

    Assumes rows are ordered by cycle/time index.
    """
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    new_cols = {}
    for col in numeric_cols:
        roll = df[col].rolling(window=window, min_periods=1).mean()
        new_name = f"{col}_{suffix}"
        new_cols[new_name] = roll

    result = pd.DataFrame(new_cols, index=df.index)
    return result


def add_delta_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add first-order differences for numeric columns."""
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    diffs = df[numeric_cols].diff().fillna(0)
    diffs = diffs.add_suffix("_delta")
    return diffs


def build_group_signals(df: pd.DataFrame, feat_index: pd.DataFrame) -> pd.DataFrame:
    """Aggregate sensors by functional group using features_index.

    For each group we compute:
    - mean
    - std
    - min
    - max
    """
    required_cols = {"column", "group"}
    if not required_cols.issubset(feat_index.columns):
        raise ValueError("features_index must have 'column' and 'group' columns")

    # Keep only columns that still exist in df
    valid = feat_index[feat_index["column"].isin(df.columns)].copy()

    group_map = valid.set_index("column")["group"].to_dict()

    # Group columns
    grouped = df[valid["column"]].groupby(group_map, axis=1)

    means = grouped.mean().add_suffix("_group_mean")
    stds = grouped.std().add_suffix("_group_std")
    mins = grouped.min().add_suffix("_group_min")
    maxs = grouped.max().add_suffix("_group_max")

    out = pd.concat([means, stds, mins, maxs], axis=1)
    return out


## 4. Building All Feature Blocks

We construct the engineered dataset in stages:

1. **Base features** (original signals)
2. **Rolling features** (temporal smoothing)
3. **Delta features** (first-order differences)
4. **Group features** (aggregated by functional sensor groups)

Finally, all feature blocks are concatenated horizontally.


In [6]:
X_base = X.copy()
print("Baseline feature table shape:", X_base.shape)


Baseline feature table shape: (2205, 43646)


In [7]:
X_roll = add_rolling_features(X_base, window=5, suffix="roll5")
print("Rolling features shape:", X_roll.shape)
display(X_roll.head(3))


Rolling features shape: (2205, 43646)


,CE_t1_roll5,CE_t2_roll5,CE_t3_roll5,CE_t4_roll5,CE_t5_roll5,CE_t6_roll5,CE_t7_roll5,CE_t8_roll5,CE_t9_roll5,CE_t10_roll5,...,VS1_t51_roll5,VS1_t52_roll5,VS1_t53_roll5,VS1_t54_roll5,VS1_t55_roll5,VS1_t56_roll5,VS1_t57_roll5,VS1_t58_roll5,VS1_t59_roll5,VS1_t60_roll5
0,47.202000,47.273000,47.2500,47.332000,47.213000,47.372000,47.273000,47.438,46.6910,46.599000,...,0.554000,0.552000,0.545000,0.553000,0.553000,0.539000,0.544,0.545000,0.535,0.5430
1,38.205000,38.047500,38.0275,38.127000,37.902000,38.007500,37.744500,37.625,37.1025,37.040000,...,0.554500,0.549500,0.546500,0.548500,0.544500,0.540500,0.542,0.539000,0.533,0.5385
2,33.321333,33.205333,33.1940,33.087333,32.948667,33.022333,32.715667,32.734,32.3710,32.146333,...,0.550667,0.547667,0.545333,0.550333,0.544333,0.541667,0.543,0.540667,0.532,0.5370


In [8]:
X_delta = add_delta_features(X_base)
print("Delta features shape:", X_delta.shape)


Delta features shape: (2205, 43646)


In [9]:
X_group = build_group_signals(X_base, features_index)
print("Group-level summary block shape:", X_group.shape)
display(X_group.head(3))


/tmp/ipython-input-3352673112.py:44: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  grouped = df[valid["column"]].groupby(group_map, axis=1)


Group-level summary block shape: (2205, 24)


,cooler_group_mean,electrical_group_mean,flow_group_mean,pressure_group_mean,temperature_group_mean,vibration_group_mean,cooler_group_std,electrical_group_std,flow_group_std,pressure_group_std,...,flow_group_min,pressure_group_min,temperature_group_min,vibration_group_min,cooler_group_max,electrical_group_max,flow_group_max,pressure_group_max,temperature_group_max,vibration_group_max
0,20.732050,2517.700666,8.507203,48.652162,36.704254,0.576950,19.473124,293.537077,2.789286,66.127115,...,0.0,0.0,30.363,0.532,47.438,2951.6,18.710,191.51,41.250,0.624
1,13.520992,2510.336011,8.559207,48.551577,37.920642,0.565850,12.374070,295.724510,2.813885,66.129864,...,0.0,0.0,33.648,0.524,29.208,2947.4,18.712,191.47,42.105,0.626
2,11.665725,2498.866588,8.542386,48.441200,38.900337,0.576533,10.606295,295.396682,2.805113,66.044587,...,0.0,0.0,35.098,0.529,23.554,2939.8,18.698,191.41,43.039,0.662


## 5. Validation and Dataset Assembly

Before saving:
- Check shapes
- Ensure no NaNs were introduced unexpectedly
- Confirm deterministic ordering

We then concatenate all blocks horizontally.


In [10]:
X_fe = pd.concat([X_base, X_roll, X_delta, X_group], axis=1)
print("Final engineered feature table shape:", X_fe.shape)

# Basic validation
assert len(X_fe) == len(X_base) == len(y)

display(X_fe.head(3))


Final engineered feature table shape: (2205, 130962)


,CE_t1,CE_t2,CE_t3,CE_t4,CE_t5,CE_t6,CE_t7,CE_t8,CE_t9,CE_t10,...,flow_group_min,pressure_group_min,temperature_group_min,vibration_group_min,cooler_group_max,electrical_group_max,flow_group_max,pressure_group_max,temperature_group_max,vibration_group_max
0,47.202,47.273,47.250,47.332,47.213,47.372,47.273,47.438,46.691,46.599,...,0.0,0.0,30.363,0.532,47.438,2951.6,18.710,191.51,41.250,0.624
1,29.208,28.822,28.805,28.922,28.591,28.643,28.216,27.812,27.514,27.481,...,0.0,0.0,33.648,0.524,29.208,2947.4,18.712,191.47,42.105,0.626
2,23.554,23.521,23.527,23.008,23.042,23.052,22.658,22.952,22.908,22.359,...,0.0,0.0,35.098,0.529,23.554,2939.8,18.698,191.41,43.039,0.662


## 6. Export for Next Step (04_modeling.ipynb)

We save the engineered feature matrix as a single Parquet file that will be loaded in Notebook 04 for:

- Train/test split
- Model training
- Threshold optimization
- Saving models for the Streamlit/Gradio app


In [11]:
PROC_DIR.mkdir(parents=True, exist_ok=True)

X_fe.to_parquet(OUT_PATH, index=False)
print("✅ Saved engineered features to:", OUT_PATH)
print("Final rows:", len(X_fe), "| Final columns:", X_fe.shape[1])


✅ Saved engineered features to: /content/drive/MyDrive/hydraulic_dashboard/data/processed/X_features_fe.parquet
Final rows: 2205 | Final columns: 130962
